In [ ]:
import os
import subprocess

# If running in Colab, clone the repo and enter it to access reference_implementation
if 'COLAB_RELEASE_TAG' in os.environ:
    if not os.path.isdir('hetnet-traffic-forecast'):
        print('Cloning repository...')
        subprocess.check_call(['git', 'clone', 'https://github.com/Sreekar-2612/hetnet-traffic-forecast.git'])
    
    # Move into the repo directory so the notebook can find reference_implementation/
    os.chdir('/content/hetnet-traffic-forecast')
    print('Colab Environment Ready! You can now run the rest of the cells.')


# TASTF: Tier-Aware Spatiotemporal Forecasting for HetNets

This notebook implements the **TASTF** (Tier-Aware Spatiotemporal Forecasting) model for Heterogeneous Cellular Networks (HetNets). 

### Core Contribution
TASTF uses a **Heterogeneous Graph Convolutional Network** that treats Macro, Pico, and Femto base stations as distinct node types with type-specific convolutional layers via PyTorch Geometric `HeteroConv`. This approach explicitly models the hierarchy and cross-tier influence in HetNets, which is then processed by a **Transformer Encoder** to capture long-range temporal dependencies.

---

## 1. Environment Setup & Dependencies

In [1]:
# Pre-built wheels (data.pyg.org) — avoids 20+ min source builds for torch-scatter/torch-sparse.
import subprocess
import sys
import torch

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip_install("pandas", "numpy", "scikit-learn", "matplotlib")

t = torch.__version__.split("+")[0]
cuda = ("cu" + torch.version.cuda.replace(".", "")) if torch.version.cuda else "cpu"
index_url = f"https://data.pyg.org/whl/torch-{t}+{cuda}.html"
print(f"Installing PyG extensions from: torch-{t}+{cuda}")

try:
    pip_install("torch-scatter", "torch-sparse", "-f", index_url)
except subprocess.CalledProcessError:
    # If your exact torch patch isn't published, try X.Y.0 wheels (same major.minor).
    parts = t.split(".")
    if len(parts) >= 3:
        t_alt = f"{parts[0]}.{parts[1]}.0"
        index_url = f"https://data.pyg.org/whl/torch-{t_alt}+{cuda}.html"
        print(f"Retrying with: torch-{t_alt}+{cuda}")
        pip_install("torch-scatter", "torch-sparse", "-f", index_url)
    else:
        raise

pip_install("torch-geometric")
print("Done.")

Installing PyG extensions from: torch-2.10.0+cu128
Done.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.data import HeteroData

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cuda


## 2. Training (canonical pipeline)

Same as `reference_implementation/train.py`: baselines, chronological split, temporal features, inverse metrics.

**Dataset (local):** put Milan `sms-call-internet-mi-*.txt` files in **`hetnet-traffic-forecast/Wireless Dataset/`** (next to `reference_implementation/`).  
Override with env `TASTF_DATA_DIR` or Colab upload path if needed.


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root():
    # Repo root: contains reference_implementation/train.py and Wireless Dataset/
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents[:8]]:
        if (base / "reference_implementation" / "train.py").is_file():
            return base
    return None

REPO = find_repo_root()
if REPO is None:
    raise FileNotFoundError(
        "Could not find hetnet-traffic-forecast. Clone/open the repo so "
        "reference_implementation/train.py exists (Colab: run first cell, then cwd should be .../hetnet-traffic-forecast)."
    )

sys.path.insert(0, str(REPO / "reference_implementation"))

from train import run_training
from paths import resolve_wireless_dataset_dir

# Prefer local folder: hetnet-traffic-forecast/Wireless Dataset
local_data = REPO / "Wireless Dataset"
if os.environ.get("TASTF_DATA_DIR"):
    DATA_DIR = os.environ["TASTF_DATA_DIR"]
elif local_data.is_dir() and any(local_data.glob("*.txt")):
    DATA_DIR = str(local_data.resolve())
    print("Using local Wireless Dataset:", DATA_DIR)
else:
    DATA_DIR = resolve_wireless_dataset_dir(None)
    print("Resolved DATA_DIR:", DATA_DIR)

run_training(data_dir=DATA_DIR, epochs=50, batch_size=32, seed=42)


## 3. Metrics and figures

Requires `reference_implementation/results.npz` after training (run from repo context below).


In [ ]:
import os
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
for base in [REPO, *REPO.parents[:8]]:
    if (base / "reference_implementation" / "read_metrics.py").is_file():
        os.chdir(base / "reference_implementation")
        if str(base / "reference_implementation") not in sys.path:
            sys.path.insert(0, str(base / "reference_implementation"))
        break
else:
    raise FileNotFoundError("reference_implementation not found; run training cell first.")

import read_metrics
read_metrics.main("results.npz")

from evaluate import plot_tier_predictions
plot_tier_predictions("results.npz", "tier_predictions.png")

from viz import plot_error_map_from_npz
plot_error_map_from_npz("results.npz", "spatial_error_map.png")
